# Unsupervised Segmentation for Remote Sensing
**DINOv2 + K-means pseudo-labels + Mean Teacher self-training**

Paper pipeline:
```
DINOv2 features → K-means init → warm-up → iterative self-training (Mean Teacher)
```

> **Runtime:** GPU (T4/V100/A100). *Runtime → Change runtime type → GPU*

## 0 · Environment

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result.stdout.strip()

# GPU check
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU found. Training will be slow on CPU.')

In [ ]:
%%capture
!pip install torchgeo omegaconf scikit-learn scipy tqdm -q

## 1 · Clone repository

In [ ]:
import os

REPO = 'https://github.com/MrDaila007/NNDP-Lab.git'
PROJECT = '/content/NNDP-Lab'

if not os.path.exists(PROJECT):
    !git clone {REPO} {PROJECT}
else:
    !git -C {PROJECT} pull

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print('Working directory:', os.getcwd())

## 2 · Download LoveDA dataset

In [ ]:
from torchgeo.datasets import LoveDA

DATA_ROOT = f'{PROJECT}/data/LoveDA'
os.makedirs(DATA_ROOT, exist_ok=True)

print('Downloading train split (~3.8 GB)...')
LoveDA(root=DATA_ROOT, split='train', download=True)
print('Downloading val split (~2.3 GB)...')
LoveDA(root=DATA_ROOT, split='val', download=True)
print('Done.')

In [ ]:
# Verify
!python tests/verify_loveda.py

## 3 · Configuration

In [ ]:
from omegaconf import OmegaConf

cfg = OmegaConf.load('configs/default.yaml')

# ── Tune these ──────────────────────────────────────────────────────
cfg.data.root         = DATA_ROOT
cfg.data.image_size   = 448      # 448 recommended for GPU; 224 for CPU
cfg.data.batch_size   = 16       # reduce if OOM
cfg.data.num_workers  = 2

cfg.model.backbone        = 'dinov2_vitb14'   # vits14 / vitb14 / vitl14
cfg.model.frozen_backbone = True

cfg.self_training.warmup_epochs    = 5
cfg.self_training.n_rounds         = 5
cfg.self_training.epochs_per_round = 10
cfg.self_training.unfreeze_round   = 3

cfg.logging.save_dir = f'{PROJECT}/checkpoints'
cfg.logging.log_dir  = f'{PROJECT}/logs'
# ────────────────────────────────────────────────────────────────────

os.makedirs(cfg.logging.save_dir, exist_ok=True)
os.makedirs(cfg.logging.log_dir,  exist_ok=True)

print(OmegaConf.to_yaml(cfg))

## 4 · Training

In [ ]:
import logging, sys
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(name)s  %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)

from data.loveda import LoveDADataset
from data.transforms import WeakAugmentation, StrongAugmentation
from training.trainer import SelfTrainer

train_ds = LoveDADataset(cfg.data.root, split='train', domain='all',
                          image_size=cfg.data.image_size, return_labels=False)
val_ds   = LoveDADataset(cfg.data.root, split='val',   domain='all',
                          image_size=cfg.data.image_size, return_labels=True)

print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

In [ ]:
aug = cfg.augmentation
weak_aug   = WeakAugmentation(cfg.data.image_size, flip_prob=aug.weak.flip_prob)
strong_aug = StrongAugmentation(
    cfg.data.image_size,
    scale_range=tuple(aug.strong.scale_range),
    color_jitter=aug.strong.color_jitter,
    blur_prob=aug.strong.blur_prob,
)

trainer  = SelfTrainer(cfg, train_ds, val_ds, torch.device(device))
best_miou = trainer.train(weak_aug=weak_aug, strong_aug=strong_aug)

print(f'\n✓ Training complete. Best mIoU (Hungarian): {best_miou:.4f}')

## 5 · Evaluation

In [ ]:
import torch
from torch.utils.data import DataLoader
from models.segmentor import Segmentor
from utils.metrics import SegmentationMetrics
from data.loveda import CLASSES

model = Segmentor(
    backbone_name=cfg.model.backbone,
    in_channels=cfg.data.in_channels,
    decoder_dim=cfg.model.decoder_dim,
    num_classes=cfg.data.num_classes,
    frozen_backbone=False,
).to(device)

ckpt = torch.load(f'{cfg.logging.save_dir}/best_model.pth', map_location=device)
teacher_state = {k[len('teacher.'):]: v for k, v in ckpt.items() if k.startswith('teacher.')}
model.load_state_dict(teacher_state)
model.eval()

metrics = SegmentationMetrics(cfg.data.num_classes)
loader  = DataLoader(val_ds, batch_size=cfg.data.batch_size,
                     shuffle=False, num_workers=cfg.data.num_workers)

with torch.no_grad():
    for images, masks in loader:
        preds = model(images.to(device)).argmax(1).cpu()
        metrics.update(preds, masks)

miou_h = metrics.compute_miou(hungarian=True)
iou_pc = metrics.compute_per_class_iou(hungarian=True)
acc    = metrics.compute_pixel_accuracy()

print(f'\nmIoU (Hungarian): {miou_h:.4f}   Pixel-Acc: {acc:.4f}\n')
for cls, iou in zip(CLASSES, iou_pc):
    bar = '█' * int(iou * 30)
    print(f'  {cls:15s}  {bar:<30s}  {iou:.3f}')

## 6 · Visualisation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import linear_sum_assignment

# ── Colour palette (GT class colours) ───────────────────────────────
PALETTE = np.array([
    [255, 255, 255],   # background  — white
    [255,  70,  70],   # building    — red
    [255, 210,  50],   # road        — yellow
    [50,  100, 255],   # water       — blue
    [180, 140, 210],   # barren      — purple
    [60,  200,  60],   # forest      — green
    [255, 180, 100],   # agriculture — orange
], dtype=np.uint8)

def colorise(label_map, palette):
    """label_map: (H,W) int → (H,W,3) RGB"""
    h, w = label_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for c in range(len(palette)):
        rgb[label_map == c] = palette[c]
    return rgb

# ── Build cluster→class mapping from confusion matrix ───────────────
cm = metrics._cm
_, col_ind = linear_sum_assignment(-cm)
cluster2class = {c: col_ind[c] for c in range(len(col_ind))}

def remap(pred, mapping):
    out = np.zeros_like(pred)
    for cl, gt in mapping.items():
        out[pred == cl] = gt
    return out


# ── Plot N random validation samples ────────────────────────────────
N_SHOW = 4
indices = np.random.choice(len(val_ds), N_SHOW, replace=False)

fig, axes = plt.subplots(N_SHOW, 3, figsize=(13, N_SHOW * 4))
fig.suptitle('Unsupervised segmentation results', fontsize=14, fontweight='bold')

col_titles = ['Input image', 'Ground truth', 'Prediction (Hungarian)']
for ax, t in zip(axes[0], col_titles):
    ax.set_title(t, fontsize=11)

for row, idx in enumerate(indices):
    img_t, mask_t = val_ds[idx]

    # Denormalise RGB
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = (img_t.numpy().transpose(1, 2, 0) * std + mean).clip(0, 1)

    with torch.no_grad():
        pred_t = model(img_t.unsqueeze(0).to(device)).argmax(1).squeeze().cpu().numpy()

    gt   = mask_t.numpy()
    gt_vis = colorise(np.where(gt == 255, 0, gt), PALETTE)

    pred_mapped = remap(pred_t, cluster2class)
    pred_vis    = colorise(pred_mapped, PALETTE)

    for ax in axes[row]:
        ax.axis('off')
    axes[row, 0].imshow(img)
    axes[row, 1].imshow(gt_vis)
    axes[row, 2].imshow(pred_vis)

# Legend
patches = [mpatches.Patch(color=PALETTE[i]/255, label=CLASSES[i])
           for i in range(len(CLASSES))]
fig.legend(handles=patches, loc='lower center', ncol=7,
           bbox_to_anchor=(0.5, -0.02), fontsize=9)
plt.tight_layout()
plt.savefig('segmentation_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved → segmentation_results.png')

In [ ]:
# ── Per-class IoU bar chart ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors  = [PALETTE[i]/255 for i in range(len(CLASSES))]
bars    = ax.barh(CLASSES, iou_pc, color=colors, edgecolor='gray', linewidth=0.5)

for bar, v in zip(bars, iou_pc):
    ax.text(v + 0.005, bar.get_y() + bar.get_height()/2,
            f'{v:.3f}', va='center', fontsize=9)

ax.axvline(miou_h, color='black', linestyle='--', linewidth=1.2, label=f'mIoU = {miou_h:.3f}')
ax.set_xlim(0, 1.05)
ax.set_xlabel('IoU')
ax.set_title('Per-class IoU (Hungarian matching)')
ax.legend()
plt.tight_layout()
plt.savefig('per_class_iou.png', dpi=120)
plt.show()
print('Saved → per_class_iou.png')

## 7 · Save artefacts to Google Drive (optional)

In [ ]:
# Uncomment to mount Drive and copy checkpoint + figures

# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_DIR = '/content/drive/MyDrive/NNDP-Lab'
# os.makedirs(DRIVE_DIR, exist_ok=True)

# import shutil
# shutil.copy(f'{cfg.logging.save_dir}/best_model.pth', DRIVE_DIR)
# shutil.copy('segmentation_results.png', DRIVE_DIR)
# shutil.copy('per_class_iou.png',        DRIVE_DIR)
# print('Copied to Drive:', DRIVE_DIR)